# What Makes a Song Popular? Spotify Tracks Analysis

This notebook explores a Spotify dataset of ~114,000 tracks across 125 genres.
The goal is to find out how popular songs (popularity > 80) differ from the rest
in their audio features, such as danceability, energy and tempo.

Data source: [Spotify Tracks Dataset on Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)

In [73]:
import pandas as pd

df = pd.read_csv("../data/tracks.csv")
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# 1. Loading and inspecting the data

The dataset has 114,000 rows and 21 columns. Each row is one track in one genre.
Popularity is a score from 0 to 100.

In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [75]:
df.describe()

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.000000,114000.000000,1.140000e+05,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000
mean,56999.500000,33.238535,2.280292e+05,0.566800,0.641383,5.309140,-8.258960,0.637553,0.084652,0.314910,0.156050,0.213553,0.474068,122.147837,3.904035
std,32909.109681,22.305078,1.072977e+05,0.173542,0.251529,3.559987,5.029337,0.480709,0.105732,0.332523,0.309555,0.190378,0.259261,29.978197,0.432621
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,28499.750000,17.000000,1.740660e+05,0.456000,0.472000,2.000000,-10.013000,0.000000,0.035900,0.016900,0.000000,0.098000,0.260000,99.218750,4.000000
50%,56999.500000,35.000000,2.129060e+05,0.580000,0.685000,5.000000,-7.004000,1.000000,0.048900,0.169000,0.000042,0.132000,0.464000,122.017000,4.000000
75%,85499.250000,50.000000,2.615060e+05,0.695000,0.854000,8.000000,-5.003000,1.000000,0.084500,0.598000,0.049000,0.273000,0.683000,140.071000,4.000000
max,113999.000000,100.000000,5.237295e+06,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000,5.000000


In [77]:
average_popularity = df["popularity"].mean()
print(f"Average track popularity: {average_popularity}")

Average track popularity: 33.2385350877193


In [79]:
max_duration_ms = df["duration_ms"].max()
max_duration_min = max_duration_ms / 60000
print(f"Maximum track duration: {max_duration_min} min")

Maximum track duration: 87.28825 min


# 2. Data cleaning

Missing values: one row had no artist, album or track name. It also had a duration of 0 ms, so it was an incomplete record and I removed it.
Unnecessary column: `Unnamed: 0` only contained row numbers, so I dropped it.
Zero duration: after removing that row, no other tracks had a duration of 0.

In [91]:
df.isna().sum()

Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [92]:
df[df["artists"].isna()]

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,...,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [93]:
df = df.dropna()

In [94]:
df.shape

(113999, 21)

In [95]:
df = df.drop(columns=["Unnamed: 0"])

In [96]:
df.shape

(113999, 20)

In [97]:
len(df[df["duration_ms"] == 0])

0

In [98]:
len(df[df["duration_ms"] > 1800000])

36

In [99]:
df[df["duration_ms"] > 1800000].sort_values("duration_ms", ascending=False)

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
73617,3Cnz3Bu9Wcw8p3kiBTXTxp,Tale Of Us,Unity (Voyage Mix),Unity (Voyage Mix) Pt. 1,35,5237295,False,0.6950,0.736000,5,-11.371,0,0.0374,0.003990,0.860000,0.0910,0.05090,124.001,4,minimal-techno
10935,0LBIf9EFHuxmuNig4JHGZo,Timo Maas,Crossing Wires 002 - Compiled And Mixed By Tim...,Crossing Wires 002 - Continuous DJ Mix,11,4789026,False,0.6970,0.632000,1,-12.469,1,0.0451,0.008150,0.871000,0.0998,0.15100,121.055,4,breakbeat
10984,0LBIf9EFHuxmuNig4JHGZo,Timo Maas,Crossing Wires 002 - Compiled And Mixed By Tim...,Crossing Wires 002 - Continuous DJ Mix,11,4789026,False,0.6970,0.632000,1,-12.469,1,0.0451,0.008150,0.871000,0.0998,0.15100,121.055,4,breakbeat
24348,6wypxnAvmv5zYewUX3VFDd,Seth Troxler,Seth Troxler - The Lab 03,The Lab 03 - Continuous DJ Mix Part 1,8,4730302,False,0.6840,0.656000,5,-11.163,0,0.0633,0.019100,0.848000,0.1030,0.30100,123.180,4,detroit-techno
73840,6JbQr97EMFD2D3Ek6gxgt1,Loco Dice,Amnesia Ibiza - Underground 10,Amnesia Ibiza Underground 10 DJ Mix,17,4563897,False,0.7830,0.728000,10,-10.176,0,0.0612,0.001580,0.840000,0.1580,0.18900,125.044,4,minimal-techno
13344,3gFme8BdfTtFHxKmuTnSbj,Mark Farina,House of OM (DJ Mix),House of Om - Mark Farina - Continuous Mix,11,4447520,False,0.8610,0.805000,11,-6.461,0,0.0711,0.005480,0.577000,0.1350,0.54500,129.306,4,chicago-house
13245,13hQAhg1owjTpTcI9xQc6c,Mark Farina,Live In Tokyo,Live In Tokyo - Continuous Mix,11,4339826,False,0.8060,0.582000,1,-12.181,1,0.0874,0.008300,0.237000,0.3280,0.68600,128.368,4,chicago-house
13195,3se0UYprtkHthJEBuk1A5K,Mark Farina,Greenhouse Construction,Greenhouse Construction,12,4334721,False,0.8460,0.744000,10,-9.260,0,0.0611,0.003710,0.335000,0.3130,0.53200,125.434,4,chicago-house
27926,6eTDnsdPlRUyrTHqFj3l7W,Lenzman;Dan Stezo,NQ State of Mind,"NQ State of Mind, Vol. 1 - Continuous DJ Mix",15,4246206,True,0.5930,0.924000,1,-3.979,1,0.0899,0.003690,0.022100,0.5340,0.24200,173.938,4,drum-and-bass
101390,2QfFLpSGF1T1pY6tq4kD7Z,Ocean Sounds,Ocean Waves Sounds to Relax and Sleep,Ocean Waves Sounds,39,4120258,False,0.0797,0.995000,1,-17.379,1,0.0612,0.932000,0.562000,0.3100,0.00001,84.788,3,sleep


In [100]:
len(df)
df["track_id"].nunique()

89740

In [101]:
df[df["track_id"] == "1PcyypJ70gm6CxTKQvCQiX"]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
25798,1PcyypJ70gm6CxTKQvCQiX,Michael Wendler,Der ultimative Wendler Hitmix,Der ultimative Wendler Hitmix XL,30,2959346,False,0.608,0.919,10,-4.071,1,0.036,0.05,0.000051,0.139,0.727,130.559,4,disco
78385,1PcyypJ70gm6CxTKQvCQiX,Michael Wendler,Der ultimative Wendler Hitmix,Der ultimative Wendler Hitmix XL,30,2959346,False,0.608,0.919,10,-4.071,1,0.036,0.05,0.000051,0.139,0.727,130.559,4,party


In [102]:
unique_tracks = df.drop_duplicates(subset="track_id")

In [103]:
len(unique_tracks)

89740

In [104]:
popular_u = unique_tracks[unique_tracks["popularity"] > 80]

In [105]:
rest_u = unique_tracks[unique_tracks["popularity"] <= 80]

In [106]:
len(popular_u) + len(rest_u)

89740

In [107]:
popular_u["danceability"].mean()

np.float64(0.65971)

In [108]:
rest_u["danceability"].mean()

np.float64(0.5616198307933662)

In [109]:
popular_u["danceability"].mean()

np.float64(0.65971)

In [110]:
rest_u["danceability"].mean()

np.float64(0.5616198307933662)